# SPARQL

**Domain:** Symbolic AI & Logic  ·  **from study list**  ·  **runnable:** yes  ·  _rdflib_

A self-contained refresher on **SPARQL** — the W3C standard query language for RDF graphs.
If SQL is how you ask questions of tables, SPARQL is how you ask questions of a **graph of
triples**: you sketch a pattern with variables and the engine finds every subgraph that fits.

## 1. What & Why

**SPARQL** ("SPARQL Protocol And RDF Query Language", pronounced *sparkle*) is the W3C standard
for querying and updating **RDF** data — the subject–predicate–object triples produced by
[`rdflib`](rdflib.ipynb), [knowledge graphs](knowledge-graphs.ipynb), and [OWL](owl.ipynb)
ontologies. It is to the Semantic Web what SQL is to relational databases.

**The problem it solves.** RDF data has no fixed schema and no tables — it's just a pile of
edges. You can't `SELECT ... FROM table` because there is no table. SPARQL lets you describe the
**shape of the connections you're looking for** ("people who know someone who works at a company
in Berlin") as a *graph pattern* full of variables, and returns every binding of those variables
that the data supports. The same query works whether the graph has 100 edges or 100 million, and
whether it lives in memory, in a triplestore, or behind an HTTP endpoint across the internet.

**Reach for it when:**
- Your data is RDF / a knowledge graph and you need flexible, ad-hoc queries.
- You want to **join across datasets** that share URIs (the whole point of *Linked Data*) —
  including remote endpoints like Wikidata or DBpedia via `SERVICE` (federated queries).
- You need graph-shaped queries: variable-length paths, optional edges, "does this exist?".

**Don't reach for it when** your data is naturally tabular (use SQL), when you need the
analytics/traversal primitives of a property graph (Cypher/Gremlin on Neo4j fit better), or when
you just need to *mutate* a handful of triples in code (manipulate the `rdflib` graph directly).

## 2. Mental Model

**SPARQL is pattern matching by example over a graph.**

Write down the triples you *wish* existed, replacing the unknowns with variables (`?x`). The
engine treats that as a template and slides it over the data, returning every assignment of
variables that makes the template a real subgraph. A query with three triple patterns sharing a
variable is exactly a **3-way join** — the shared variable forces the matches to line up.

```
The data (triples):                A query pattern (a Basic Graph Pattern):

  :alice  :knows  :bob              ?p   :knows  ?friend .
  :alice  :age    30                ?p   :age    ?a      .
  :bob    :age    25                FILTER(?a > 26)
  :bob    :knows  :carol
                                    matches ?p=:alice, ?friend=:bob, ?a=30
                                    (the only person over 26 who knows someone)
```

Every shared variable is a join key; `FILTER` prunes rows after matching; `OPTIONAL` is a left
join (keep the row even if the optional edge is missing). That's 90% of SPARQL.

## 3. Key Concepts

| Term | What it is |
|---|---|
| **Triple pattern** | A statement with variables: `?s :knows ?o`. The atom of a query. |
| **Basic Graph Pattern (BGP)** | A set of triple patterns; shared variables = joins. Matched conjunctively (all must hold). |
| **Variable** | `?name` (or `$name`). Bound to a node (URI, literal, or blank node) by matching. |
| **Prefix** | `PREFIX foaf: <http://xmlns.com/foaf/0.1/>` — a shorthand for long URI namespaces. |
| **Query form** | `SELECT` (table of bindings), `ASK` (yes/no), `CONSTRUCT` (build a new graph), `DESCRIBE` (let the server return relevant triples). |
| **`FILTER`** | A boolean test that drops non-matching solutions: `FILTER(?age > 18 && regex(?n, "^A"))`. |
| **`OPTIONAL`** | Left join — bind extra variables if present, leave them unbound otherwise. |
| **`UNION`** | Match either of two patterns (disjunction). |
| **`BIND` / `VALUES`** | `BIND(?a*2 AS ?b)` computes a value; `VALUES ?x { :a :b }` injects an inline table. |
| **Aggregates** | `GROUP BY` + `COUNT/SUM/AVG/MIN/MAX/SAMPLE/GROUP_CONCAT`, with `HAVING` to filter groups. |
| **Property path** | Regex over predicates: `foaf:knows+` (one-or-more hops), `a/rdfs:subClassOf*`, `^p` (inverse), `p1|p2`. |
| **Solution modifiers** | `ORDER BY`, `LIMIT`, `OFFSET`, `DISTINCT`, `REDUCED`. |
| **SPARQL Update** | The write half: `INSERT DATA`, `DELETE DATA`, `DELETE/INSERT ... WHERE`. |
| **`SERVICE`** | Federation — run part of the query against a *remote* endpoint and join the results. |
| **Protocol** | SPARQL is also an HTTP protocol: POST a query to an endpoint URL, get JSON/XML/CSV back. |

## 4. Setup

Any RDF stack works; we use **[rdflib](https://rdflib.readthedocs.io/)**, which ships a complete
**SPARQL 1.1 engine that runs fully in-memory and offline** — no server, no network. That makes
every example below reproducible in a fresh kernel.

```bash
pip install rdflib            # in-memory graph + SPARQL engine (used here)
pip install SPARQLWrapper     # optional: convenience client for remote endpoints
```

For querying remote endpoints (Wikidata, DBpedia) you only need an internet connection — those
last examples are gated behind an `os.getenv` check so the notebook still runs offline.

In [1]:
# %pip install rdflib
from importlib.metadata import version
import rdflib

print("rdflib", version("rdflib"))
print("SPARQL engine: in-memory, offline — no server required")

rdflib 7.6.0
SPARQL engine: in-memory, offline — no server required


## 5. Worked Examples

### Example 1 — Load a graph and run a `SELECT`

We build a tiny social graph in **Turtle** (a compact RDF text format), then ask the bread-and-
butter question: a BGP with a join (`?p :knows ?f`, `?f :age ?age`), a `FILTER`, and ordering.

In [2]:
from rdflib import Graph

DATA = """
@prefix : <http://example.org/> .
@prefix foaf: <http://xmlns.com/foaf/0.1/> .

:alice  a foaf:Person ; foaf:name "Alice" ; :age 30 ; :knows :bob, :carol .
:bob    a foaf:Person ; foaf:name "Bob"   ; :age 25 ; :knows :carol .
:carol  a foaf:Person ; foaf:name "Carol" ; :age 41 ; :knows :alice .
:dave   a foaf:Person ; foaf:name "Dave"  ; :age 19 .
"""

g = Graph()
g.parse(data=DATA, format="turtle")
print("triples loaded:", len(g))

q = """
PREFIX :     <http://example.org/>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>

SELECT ?name ?fname ?fage
WHERE {
    ?p     :knows ?f .
    ?p     foaf:name ?name .
    ?f     foaf:name ?fname .
    ?f     :age   ?fage .
    FILTER(?fage >= 30)          # only friends aged 30+
}
ORDER BY DESC(?fage)
"""

for row in g.query(q):
    print(f"{row.name} knows {row.fname} (age {row.fage})")

triples loaded: 16
Alice knows Carol (age 41)
Bob knows Carol (age 41)
Carol knows Alice (age 30)


### Example 2 — `OPTIONAL`, aggregation, `ASK`, and a property path

Four idioms on the same graph:

- **`OPTIONAL`** (left join): list everyone, showing how many people each one knows — including
  Dave, who knows nobody, via `COUNT` over an optional edge.
- **`GROUP BY` + aggregate** with `HAVING`.
- **`ASK`**: a boolean existence check.
- **Property path** `:knows+`: reachability over one-or-more `:knows` hops (a transitive
  closure you'd otherwise need recursion for).

In [3]:
# --- Aggregate: how many people does each person know? (OPTIONAL keeps Dave) ---
q_agg = """
PREFIX :     <http://example.org/>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
SELECT ?name (COUNT(?f) AS ?n)
WHERE {
    ?p foaf:name ?name .
    OPTIONAL { ?p :knows ?f }
}
GROUP BY ?p ?name
ORDER BY DESC(?n) ?name
"""
print("knows-count per person:")
for row in g.query(q_agg):
    print(f"  {row.name}: {int(row.n)}")

# --- ASK: is there anyone over 40? ---
q_ask = """
PREFIX : <http://example.org/>
ASK { ?p :age ?a . FILTER(?a > 40) }
"""
print("\nanyone over 40? ->", bool(g.query(q_ask)))

# --- Property path: who can Alice reach via :knows+ (transitive)? ---
q_path = """
PREFIX :     <http://example.org/>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
SELECT DISTINCT ?reached
WHERE {
    :alice :knows+ ?x .
    ?x foaf:name ?reached .
}
ORDER BY ?reached
"""
print("\nreachable from Alice via knows+:",
      [str(r.reached) for r in g.query(q_path)])

knows-count per person:
  Alice: 2
  Bob: 1
  Carol: 1
  Dave: 0

anyone over 40? -> True

reachable from Alice via knows+: ['Alice', 'Bob', 'Carol']


### Example 3 — `CONSTRUCT` (query that returns a *graph*) and `SPARQL Update`

`SELECT` returns a table; **`CONSTRUCT`** returns new triples — handy for transforming or
materializing a view. **Update** (`INSERT`/`DELETE`) mutates the graph in place.

In [4]:
# CONSTRUCT a symmetric "peer" graph: if A knows B, emit A :peerOf B.
q_construct = """
PREFIX : <http://example.org/>
CONSTRUCT { ?a :peerOf ?b }
WHERE     { ?a :knows  ?b }
"""
peers = g.query(q_construct).graph          # CONSTRUCT yields a Graph
print("constructed peer triples:", len(peers))
print(peers.serialize(format="turtle").strip()[:200], "...\n")

# SPARQL Update: add a triple, then delete one. g.update() mutates g in place.
before = len(g)
g.update("""
PREFIX : <http://example.org/>
INSERT DATA { :dave :knows :alice }
""")
g.update("""
PREFIX : <http://example.org/>
DELETE DATA { :bob :knows :carol }
""")
print(f"triples before update: {before}, after insert+delete: {len(g)}")

# Confirm the change took effect.
ask_dave = g.query("""PREFIX : <http://example.org/>
                      ASK { :dave :knows :alice }""")
print("dave now knows alice? ->", bool(ask_dave))

constructed peer triples: 4
@prefix ns1: <http://example.org/> .

ns1:alice ns1:peerOf ns1:bob,
        ns1:carol .

ns1:bob ns1:peerOf ns1:carol .

ns1:carol ns1:peerOf ns1:alice . ...

triples before update: 16, after insert+delete: 16
dave now knows alice? -> True


### Example 4 — Querying a *remote* endpoint (gated, optional)

The examples above are offline. Real Linked Data lives behind public SPARQL endpoints
(Wikidata, DBpedia). This is the **SPARQL protocol**: POST a query string to an endpoint URL.
We gate it behind `RUN_SPARQL_NETWORK` so the notebook still runs top-to-bottom offline, while
still showing the call shape.

In [5]:
import os

# Wikidata: 3 chemical elements and their symbols. Runs only when you opt in.
WIKIDATA = "https://query.wikidata.org/sparql"
REMOTE_Q = """
SELECT ?elementLabel ?symbol WHERE {
    ?element wdt:P31 wd:Q11344 ;        # instance of: chemical element
             wdt:P246 ?symbol .         # element symbol
    SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
}
ORDER BY ?symbol
LIMIT 3
"""

if os.getenv("RUN_SPARQL_NETWORK"):
    from rdflib.plugins.stores.sparqlstore import SPARQLStore
    from rdflib import Graph as RemoteGraph
    rg = RemoteGraph(store=SPARQLStore(WIKIDATA, returnFormat="json"))
    for row in rg.query(REMOTE_Q):
        print(f"{row.elementLabel}: {row.symbol}")
else:
    print("RUN_SPARQL_NETWORK not set — skipping live Wikidata call.")
    print("Set it (and `pip install rdflib`) to execute this query against:")
    print(" ", WIKIDATA)
    print("Query shape:", " ".join(REMOTE_Q.split())[:90], "...")

RUN_SPARQL_NETWORK not set — skipping live Wikidata call.
Set it (and `pip install rdflib`) to execute this query against:
  https://query.wikidata.org/sparql
Query shape: SELECT ?elementLabel ?symbol WHERE { ?element wdt:P31 wd:Q11344 ; # instance of: chemical  ...


## 6. Gotchas & Pitfalls

- **Forgetting the trailing `.`** between triple patterns, or putting a `;` (same subject) vs `,`
  (same subject *and* predicate) — a classic Turtle/SPARQL syntax trap.
- **`FILTER` doesn't bind, it prunes.** `FILTER(?x = :alice)` keeps rows where `?x` is already
  bound to `:alice`; it cannot *introduce* `?x`. To bind a value use `VALUES` or a triple
  pattern. Also: a `FILTER` inside `{}` applies to the whole group, regardless of where you put it.
- **`OPTIONAL` ordering matters.** A `FILTER` referencing an optional variable behaves
  differently inside vs outside the `OPTIONAL` block; misplacing it silently changes results.
- **Datatype/literal mismatches.** `"30"` (string) ≠ `30` (integer) ≠ `"30"^^xsd:int`. A
  `FILTER(?age > 18)` over string-typed ages compares lexically or fails. Type your literals.
- **Unbounded queries melt servers.** No `LIMIT` + a property path like `?x rdfs:subClassOf* ?y`
  over a huge graph = timeout. Public endpoints enforce time/row limits and will cut you off.
- **`DISTINCT` is not free.** It forces full materialization; on big result sets it's expensive.
  Use `REDUCED` if you only want *fewer* duplicates, not *zero*.
- **`CONSTRUCT` results are a graph, not rows.** In rdflib, use `.graph` (or iterate triples) —
  treating it like a `SELECT` result gives you triple tuples, not named bindings.
- **Open-world surprises.** SPARQL queries what's *asserted*. It does **not** run a reasoner — if
  `:Dog rdfs:subClassOf :Animal` and you query for `:Animal` instances, you get nothing unless
  the store does inference or you spell out the path (`a/rdfs:subClassOf* :Animal`).
- **Federation (`SERVICE`) is fragile.** Remote endpoints rate-limit, time out, and differ in
  supported features; always `LIMIT` and expect partial failures.

## 7. When to Use vs Alternatives

| Option | Best at | Trade-off vs SPARQL |
|---|---|---|
| **SPARQL / RDF** | Standardized, schema-flexible queries over RDF; joining Linked Data across the web via shared URIs; W3C interoperability. | Verbose; RDF's triple model + open-world semantics have a learning curve; analytics are awkward. |
| **SQL / relational** | Tabular data with a fixed schema; transactions; mature optimizers and tooling. | Rigid schema; multi-hop joins get painful; no global identifiers across databases. |
| **Cypher / Gremlin** (property graphs, Neo4j) | Ergonomic traversals, variable-length paths, graph analytics; properties on edges. | Not a W3C standard (Cypher is vendor-ish); no built-in global URIs / Linked-Data federation. |
| **Document / KV (Mongo, etc.)** | Denormalized nested docs, simple lookups, high write throughput. | No joins to speak of; relationships are your problem, not the engine's. |
| **In-code graph traversal** (`rdflib` API, NetworkX) | A few triples, or imperative one-off logic. | Reinvents joins/filters/paths that SPARQL gives you declaratively; doesn't scale or federate. |

**Rule of thumb:** if your data is (or should be) **RDF / Linked Data**, or you need to query
across heterogeneous datasets that share identifiers, SPARQL is the standard answer. If it's a
single app's tabular data, use SQL. If it's a richly-connected property graph you mostly traverse
locally, a property-graph engine is more ergonomic. SPARQL also pairs with
[OWL](owl.ipynb)/[reasoners](description-logic-reasoners.ipynb): reason first to materialize
inferences, then query the enriched graph.

## 8. Resources

- **SPARQL 1.1 Query Language (W3C Recommendation)** — the spec, surprisingly readable:
  https://www.w3.org/TR/sparql11-query/
- **SPARQL 1.1 Update** (the write half — INSERT/DELETE):
  https://www.w3.org/TR/sparql11-update/
- **rdflib — Querying with SPARQL** (the engine used in this notebook):
  https://rdflib.readthedocs.io/en/stable/intro_to_sparql.html
- **Wikidata Query Service** — a live endpoint with a great example gallery to learn from:
  https://query.wikidata.org/ (examples: https://www.wikidata.org/wiki/Wikidata:SPARQL_query_service/queries/examples)
- **"Learning SPARQL", Bob DuCharme (O'Reilly)** — the standard practical book on the language:
  https://www.learningsparql.com/